In [1]:
import requests
import json
import pandas as pd
import copy
import time
import requests
import pandas as pd
import json
import copy
import time

In [2]:
def generar_meses():
    meses = ["ene", "feb", "mar", "abr", "may", "jun",
             "jul", "ago", "sep", "oct", "nov", "dic"]
    
    resultado = []
    for anio in range(2015, 2026):
        for mes in meses:
            resultado.append(f"{mes}. {str(anio)[-2:]}")
    
    return resultado


def parse_looker_response(data, mes):
    dataset = data['dataResponse'][0]['dataSubset'][0]['dataset']['tableDataset']
    
    columns_list = dataset['column']
    total_rows = dataset['size']

    final_columns = {}

    for i, col_data in enumerate(columns_list):

        col_name = col_data.get('name') or f"col_{i}"

        key = 'stringColumn' if 'stringColumn' in col_data else 'doubleColumn'
        values = col_data.get(key, {}).get('values', [])
        null_indices = col_data.get('nullIndex', [])

        full_col = []
        val_ptr = 0

        for idx in range(total_rows):
            if idx in null_indices:
                full_col.append(None)
            else:
                full_col.append(values[val_ptr])
                val_ptr += 1

        final_columns[col_name] = full_col

    df = pd.DataFrame(final_columns)
    df["mes"] = mes

    return df


# 🔥 NUEVA FUNCIÓN
def find_mes_filter(payload):
    filters = payload['dataRequest'][0]['datasetSpec']['filters']
    
    for i, f in enumerate(filters):
        try:
            exp = f['filterDefinition']['filterExpression']
            
            if 'stringValues' in exp:
                val = exp['stringValues'][0]
                
                # detecta formato tipo "ene. 24"
                if isinstance(val, str) and "." in val:
                    return i
        except:
            continue
    
    raise ValueError("No se encontró el filtro de mes")


def run_looker_pipeline(
    cookies, 
    headers, 
    params, 
    json_data,
    pais=None,
    ciudad=None,
    operacion=None,
    columnas=None,
    mes_filter_idx=None,        # 🔥 NUEVO
    mes_field_name=None         # 🔥 OPCIONAL (ej: "_fecha_")
):
    
    meses = generar_meses()
    dfs = []

    # ==============================
    # 🔥 DETECCIÓN DEL FILTRO
    # ==============================
    if mes_filter_idx is None:
        filters = json_data['dataRequest'][0]['datasetSpec']['filters']
        
        for i, f in enumerate(filters):
            try:
                field = f['filterDefinition']['filterExpression']['queryTimeTransformation']['dataTransformation']['sourceFieldName']
                
                if mes_field_name and field == mes_field_name:
                    mes_filter_idx = i
                    break
            except:
                continue

        if mes_filter_idx is None:
            raise ValueError("No se encontró el filtro de mes")

    print(f"📍 Usando filtro de mes en índice: {mes_filter_idx}")

    # ==============================
    # LOOP
    # ==============================
    for mes in meses:
        try:
            payload = copy.deepcopy(json_data)

            payload['dataRequest'][0]['datasetSpec']['filters'][mes_filter_idx]['filterDefinition']['filterExpression']['stringValues'] = [mes]

            response = requests.post(
                'https://lookerstudio.google.com/embed/u/0/batchedDataV2',
                params=params,
                cookies=cookies,
                headers=headers,
                json=payload,
            )

            if response.status_code != 200:
                print(f"❌ Error en {mes}")
                continue

            raw_str = response.text[5:].strip()
            data = json.loads(raw_str)

            df = parse_looker_response(data, mes)

            if len(df) == 0:
                print(f"⚠️ {mes} sin datos")
                continue

            # ==============================
            # METADATOS
            # ==============================
            if pais is not None:
                df["pais"] = pais
            if ciudad is not None:
                df["ciudad"] = ciudad
            if operacion is not None:
                df["operacion"] = operacion

            # ==============================
            # RENOMBRE
            # ==============================
            if columnas is not None:
                df = df.rename(columns=columnas)

            dfs.append(df)

            print(f"✅ {mes} | {len(df)} filas")

            time.sleep(1)

        except Exception as e:
            print(f"⚠️ Error en {mes}: {e}")

    if dfs:
        return pd.concat(dfs, ignore_index=True)
    else:
        return pd.DataFrame()

#### Lima
##### Venta

In [3]:
cookies = {
    'RAP_XSRF_TOKEN': 'AImk1AIN7OCz9crvibv1OHp9Pimw-ueBgA:1774563148249',
    '__Secure-3PSID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076',
    '__Secure-3PAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    'NID': '530=gDjAWKDTV_8uQH1Ohgs2-HpHJQcITzPv6dULBY0hR6yyNB7F-2oKCtNgUIVqnBncTRuu45eOBq6_lW2D6tVU-MigHYlB0DmkJC6kWVm-wHjgui8-EbZHXGsyznuslwPcIcmzy-prrDvfd3Ru4jk4XAZIsLM2PR93WJB9vo6CIIUv3Baxc4hdYD6JqeNaUwZIGRZRYRfDB8F8L5SFRmLH4IzTGsJy8eI-2dOo8_s_lucUsmMTvyzai6sq44m9HaW6vhoWF3mmEZj0OZwJBSS4kDht1ZBaiX23cFtOK2JQziKmQCWmkmMlQxsdJWvrCLPX1nMgribsUGpjxaZh-919GqkPY3XHWXwADtC-m7xConPO6YUJLGeEVKfgt2mq5pJ7L7bhYFa_rikYJe9alMKX_qP5ZE9Pg5ygbOhHbH2HnDjvwd7vZGQFA7Zti3JnKMUA0UYAhZC2MItW6mCmdAzopnFc4MXfJpTizSs4MXedOx5Lg8uzBXgbHBSEEotmzQF-akWJIqKxsFCyYXr34cJYGGVzDabEJCo4zB4NY9txqCxzpovTfSz-J29CAqtpnPEtfe0fLoC0c82SiX2K7JOazTrPP0A3HCDvSVDQAY1ME-7kmihqvrNpmPJkgZ8pcwPJ_1p5kW285IR4Dh-WXQuBM-06SxzRpmMxaMP3d5iElNoTBIuAsJ0f0XGrU2_1GCSl4aZj3uowPiLdC5eJ7n7aW969icdLcNON9wS9p8YGWBRanBXJINsn8UxkN5OlAHQ5tmezAppFldKnaeX5BzsSKBEib_DHINUgmztxE0Xy4xYZBBU07WUVx36j-klMs8u7kp8ismKiDnSMryyzbSzBCih52um0c506UklUG0kBhsH2imhGgN4pfF2MZpp9GuQs4MbTgQOzS3jE5t_ivryStDlTbshN7m05iVvfbMqZm_fRRFsXSQKLF4Zq3aJncxYKvM8c6HL6NqU0j3h-wMMeeB77AeJePTI-8CB_zJxjTU-6BulqxurtkY7yT6y_WUoyWibop--ikipErr3HCBDTgTweQ_vRh41NQ1vvMxRiUqFsuC03OK0_S5IzByqa_HtjsaXe_e86hy1mftbM9yjPm05PCR69QkzYbfzqf75XsrXWgO6cP6OWZrFApgkSM_ROXsTbVgcpyojMCt1D_XG41CKQPnNEYRl4WRZfsdstBxggIQbLXuJfD2xpiy2QrOFP6j2QT1QakXGQEu-4q6RWdQAWmlC0T8V85lgoDw4Op-MKDOhN9IiV2I8GuRjHmWQpyfplO7fy0VjK0Qi-lRc4OW7SJMnIcWvG8CotW8Bvxk0kEKjn3zA7V0mG9B68HyF5wippZsBbHDMMbeDTnWcHftdQWOo2ss0GX3TzgiutFzOldhLZeXBDhwqQX3_0opcjwHfs2UvgRwR37x_ZE9Tp--2vvOWdhBjGU8zlo1n13XaUuZrJIxbuRTPBWcx9zq4kClqeK67slm2Y1BACt6U5r2TofmzwKNUAWneSWJQczzDly3pV1YiyuvM3YDKx1ZJzPCBYErhRlPdfFhnOxg2ol2GI41xcR1X0M_63xB9WJLHyMklcdrn3a3JMA1pvrWB7eYQKFWvobakGGA2NZArNxZ4eaH9kWuH_Bbkt0edkqT9_RzHV3BYYDw5www-CHiS79FePx2HgsqJAqlNkrmHFaTPT9isPpdwJQ9X3aan1RkSlSpYMXIjYds2KgSr5Q4fLWZk7h1U-zdvNmyF1etF_LO-eHecJzxPP_lGnQlhSWsSfLP9jl9996_o7-5SivKgWv_Q4aV0-6PKN9lupTWKQ10qSGPRcl0jA4wcw0AaTLuZfc_wkraUiEMw9n0tNQySDfxjXVzC-Fa_3mbRG1FqBQCZGA3U3932u4Es9mhf-yq7DreHX6zt2qFRGCOvpsz-7CV91y_fDJ__6pYD5A55PWtafYs3dHrziNV-8_0kiDGFLPGLrPOw_mM2BZjLTbgpWEXc5CpLjARx_oYvFU99Uqn6ljjPvwR5dPvhmL12CceVOs52AlarWKnoNV2zxzdTmbjPizwzbf4VaoAhHcOG78fL4Sjj9EbooHXbku5kfBqZGQj0axSBauC7YIEcIrAD7oiB7KLNjQ5IRUq65QiRkMoPKQvfhUcioET-V',
    '__Secure-3PSIDTS': 'sidts-CjEBWhotCQJP50JpyGJfg879V0TXwvSagsd1A9gkGTVKcoigQ8NU2W6Po8DnUR8NngGDEAA',
    '__Secure-3PSIDCC': 'AKEyXzXVD6V54t-2a0JlzxBD4_tg8YLiJBXmYFhWSs7vnGlUUXrAR2ngltr7iNa7ZkFq2oP03y6SZQ',
}

headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'es-US,es-419;q=0.9,es;q=0.8',
    'content-type': 'application/json',
    'encoding': 'null',
    'origin': 'https://lookerstudio.google.com',
    'priority': 'u=1, i',
    'referer': 'https://lookerstudio.google.com/embed/u/0/reporting/92cfc496-c528-4192-97dd-7aa3dfdc57e2/page/Ibx9',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'sec-fetch-storage-access': 'active',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-browser-channel': 'stable',
    'x-browser-copyright': 'Copyright 2026 Google LLC. All Rights reserved.',
    'x-browser-validation': 'NNdAQ4umZKwG7+IzfjmwHCgaZsw=',
    'x-browser-year': '2026',
    'x-client-data': 'CJC2yQEIo7bJAQipncoBCNmOywEIk6HLAQiFoM0BGLGKzwEY7LHPAQ==',
    'x-rap-xsrf-token': 'AImk1AIN7OCz9crvibv1OHp9Pimw-ueBgA:1774563148249',
    # 'cookie': 'RAP_XSRF_TOKEN=AImk1AIN7OCz9crvibv1OHp9Pimw-ueBgA:1774563148249; __Secure-3PSID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076; __Secure-3PAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; NID=530=gDjAWKDTV_8uQH1Ohgs2-HpHJQcITzPv6dULBY0hR6yyNB7F-2oKCtNgUIVqnBncTRuu45eOBq6_lW2D6tVU-MigHYlB0DmkJC6kWVm-wHjgui8-EbZHXGsyznuslwPcIcmzy-prrDvfd3Ru4jk4XAZIsLM2PR93WJB9vo6CIIUv3Baxc4hdYD6JqeNaUwZIGRZRYRfDB8F8L5SFRmLH4IzTGsJy8eI-2dOo8_s_lucUsmMTvyzai6sq44m9HaW6vhoWF3mmEZj0OZwJBSS4kDht1ZBaiX23cFtOK2JQziKmQCWmkmMlQxsdJWvrCLPX1nMgribsUGpjxaZh-919GqkPY3XHWXwADtC-m7xConPO6YUJLGeEVKfgt2mq5pJ7L7bhYFa_rikYJe9alMKX_qP5ZE9Pg5ygbOhHbH2HnDjvwd7vZGQFA7Zti3JnKMUA0UYAhZC2MItW6mCmdAzopnFc4MXfJpTizSs4MXedOx5Lg8uzBXgbHBSEEotmzQF-akWJIqKxsFCyYXr34cJYGGVzDabEJCo4zB4NY9txqCxzpovTfSz-J29CAqtpnPEtfe0fLoC0c82SiX2K7JOazTrPP0A3HCDvSVDQAY1ME-7kmihqvrNpmPJkgZ8pcwPJ_1p5kW285IR4Dh-WXQuBM-06SxzRpmMxaMP3d5iElNoTBIuAsJ0f0XGrU2_1GCSl4aZj3uowPiLdC5eJ7n7aW969icdLcNON9wS9p8YGWBRanBXJINsn8UxkN5OlAHQ5tmezAppFldKnaeX5BzsSKBEib_DHINUgmztxE0Xy4xYZBBU07WUVx36j-klMs8u7kp8ismKiDnSMryyzbSzBCih52um0c506UklUG0kBhsH2imhGgN4pfF2MZpp9GuQs4MbTgQOzS3jE5t_ivryStDlTbshN7m05iVvfbMqZm_fRRFsXSQKLF4Zq3aJncxYKvM8c6HL6NqU0j3h-wMMeeB77AeJePTI-8CB_zJxjTU-6BulqxurtkY7yT6y_WUoyWibop--ikipErr3HCBDTgTweQ_vRh41NQ1vvMxRiUqFsuC03OK0_S5IzByqa_HtjsaXe_e86hy1mftbM9yjPm05PCR69QkzYbfzqf75XsrXWgO6cP6OWZrFApgkSM_ROXsTbVgcpyojMCt1D_XG41CKQPnNEYRl4WRZfsdstBxggIQbLXuJfD2xpiy2QrOFP6j2QT1QakXGQEu-4q6RWdQAWmlC0T8V85lgoDw4Op-MKDOhN9IiV2I8GuRjHmWQpyfplO7fy0VjK0Qi-lRc4OW7SJMnIcWvG8CotW8Bvxk0kEKjn3zA7V0mG9B68HyF5wippZsBbHDMMbeDTnWcHftdQWOo2ss0GX3TzgiutFzOldhLZeXBDhwqQX3_0opcjwHfs2UvgRwR37x_ZE9Tp--2vvOWdhBjGU8zlo1n13XaUuZrJIxbuRTPBWcx9zq4kClqeK67slm2Y1BACt6U5r2TofmzwKNUAWneSWJQczzDly3pV1YiyuvM3YDKx1ZJzPCBYErhRlPdfFhnOxg2ol2GI41xcR1X0M_63xB9WJLHyMklcdrn3a3JMA1pvrWB7eYQKFWvobakGGA2NZArNxZ4eaH9kWuH_Bbkt0edkqT9_RzHV3BYYDw5www-CHiS79FePx2HgsqJAqlNkrmHFaTPT9isPpdwJQ9X3aan1RkSlSpYMXIjYds2KgSr5Q4fLWZk7h1U-zdvNmyF1etF_LO-eHecJzxPP_lGnQlhSWsSfLP9jl9996_o7-5SivKgWv_Q4aV0-6PKN9lupTWKQ10qSGPRcl0jA4wcw0AaTLuZfc_wkraUiEMw9n0tNQySDfxjXVzC-Fa_3mbRG1FqBQCZGA3U3932u4Es9mhf-yq7DreHX6zt2qFRGCOvpsz-7CV91y_fDJ__6pYD5A55PWtafYs3dHrziNV-8_0kiDGFLPGLrPOw_mM2BZjLTbgpWEXc5CpLjARx_oYvFU99Uqn6ljjPvwR5dPvhmL12CceVOs52AlarWKnoNV2zxzdTmbjPizwzbf4VaoAhHcOG78fL4Sjj9EbooHXbku5kfBqZGQj0axSBauC7YIEcIrAD7oiB7KLNjQ5IRUq65QiRkMoPKQvfhUcioET-V; __Secure-3PSIDTS=sidts-CjEBWhotCQJP50JpyGJfg879V0TXwvSagsd1A9gkGTVKcoigQ8NU2W6Po8DnUR8NngGDEAA; __Secure-3PSIDCC=AKEyXzXVD6V54t-2a0JlzxBD4_tg8YLiJBXmYFhWSs7vnGlUUXrAR2ngltr7iNa7ZkFq2oP03y6SZQ',
}

params = {
    'appVersion': '20260315_0000',
}

json_data = {
    'dataRequest': [
        {
            'requestContext': {
                'reportContext': {
                    'reportId': '92cfc496-c528-4192-97dd-7aa3dfdc57e2',
                    'pageId': '14728046',
                    'mode': 1,
                    'componentId': 'cd-75b23orndc',
                    'displayType': 'simple-table',
                    'actionId': 'crossFilters',
                },
                'requestMode': 0,
            },
            'datasetSpec': {
                'dataset': [
                    {
                        'datasourceId': '4f53dde5-ccd3-47a8-a13a-1683958b3094',
                        'revisionNumber': 0,
                        'parameterOverrides': [],
                    },
                ],
                'queryFields': [
                    {
                        'name': 'qt_u5wcznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_nombre_n3_',
                        },
                    },
                    {
                        'name': 'qt_vfpcznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_estrenar_precio_',
                            'aggregation': 1,
                            'transformationConfig': {
                                'transformationType': 0,
                            },
                        },
                    },
                    {
                        'name': 'qt_duucznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_index_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_euucznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_usado_precio_',
                            'aggregation': 1,
                        },
                    },
                ],
                'sortData': [
                    {
                        'sortColumn': {
                            'name': 'qt_duucznzjic',
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'dataTransformation': {
                                'sourceFieldName': '_index_precio_',
                                'aggregation': 1,
                            },
                        },
                        'sortDir': 1,
                    },
                ],
                'includeRowsCount': True,
                'relatedDimensionMask': {
                    'addDisplay': False,
                    'addUniqueId': False,
                    'addLatLong': False,
                },
                'paginateInfo': {
                    'startRow': 1,
                    'rowsCount': 1000,
                },
                'dsFilterOverrides': [],
                'filters': [
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': False,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_ho99vjcxac',
                                },
                                'filterConditionType': 'NU',
                                'stringValues': [
                                    '',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_nombre_n3_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_9p89ss7ihc',
                                },
                                'filterConditionType': 'EQ',
                                'stringValues': [
                                    'LIMA',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_ciudad_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'name': 'qt_mspdznzjic',
                                    'ns': 't0',
                                },
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_fecha_',
                                    },
                                },
                                'filterConditionType': 'IN',
                                'stringValues': [
                                    'ago 25',
                                ],
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                        'isCanvasFilter': True,
                    },
                ],
                'features': [],
                'dateRanges': [],
                'contextNsCount': 1,
                'dateRangeDimensions': [
                    {
                        'name': 'qt_vs1cznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_fecha_as_date_',
                            'transformationConfig': {
                                'transformationType': 5,
                            },
                        },
                    },
                ],
                'calculatedField': [],
                'needGeocoding': False,
                'geoFieldMask': [],
                'multipleGeocodeFields': [],
                'timezone': 'America/Mexico_City',
            },
            'role': 'main',
            'retryHints': {
                'useClientControlledRetry': True,
                'isLastRetry': False,
                'retryCount': 0,
                'originalRequestId': 'cd-75b23orndc_0_0',
            },
        },
    ],
}

In [ ]:
lima_venta = run_looker_pipeline(cookies, headers, params, json_data, pais="Peru",ciudad="Lima",operacion="sell", 
                                mes_field_name="_fecha_",
                                 
                                 columnas={"col_0":"neighborhood",
                                                                                                                                     "col_1":"new",
                                                                                                                                     "col_2": "index",
                                                                                                                                     "col_3": "used"
                                                                                                                                     })
print("Total de registros", len(lima_venta))
lima_venta.to_csv("lima_venta.csv", encoding="latin1",index=False)
lima_venta

📍 Usando filtro de mes en índice: 2
⚠️ ene. 15 sin datos
⚠️ feb. 15 sin datos
⚠️ mar. 15 sin datos
⚠️ abr. 15 sin datos
⚠️ may. 15 sin datos
⚠️ jun. 15 sin datos
⚠️ jul. 15 sin datos
⚠️ ago. 15 sin datos
⚠️ sep. 15 sin datos
⚠️ oct. 15 sin datos
⚠️ nov. 15 sin datos
⚠️ dic. 15 sin datos
⚠️ ene. 16 sin datos
⚠️ feb. 16 sin datos
⚠️ mar. 16 sin datos
⚠️ abr. 16 sin datos
⚠️ may. 16 sin datos
⚠️ jun. 16 sin datos
⚠️ jul. 16 sin datos
⚠️ ago. 16 sin datos
⚠️ sep. 16 sin datos
⚠️ oct. 16 sin datos
⚠️ nov. 16 sin datos
⚠️ dic. 16 sin datos
⚠️ ene. 17 sin datos
⚠️ feb. 17 sin datos
⚠️ mar. 17 sin datos
✅ abr. 17 | 75 filas
✅ may. 17 | 75 filas
✅ jun. 17 | 75 filas
✅ jul. 17 | 75 filas
✅ ago. 17 | 75 filas
✅ sep. 17 | 75 filas
✅ oct. 17 | 75 filas
✅ nov. 17 | 75 filas
✅ dic. 17 | 74 filas
✅ ene. 18 | 75 filas
✅ feb. 18 | 75 filas
✅ mar. 18 | 75 filas
✅ abr. 18 | 75 filas
✅ may. 18 | 75 filas
✅ jun. 18 | 75 filas
✅ jul. 18 | 75 filas
✅ ago. 18 | 75 filas
✅ sep. 18 | 75 filas
✅ oct. 18 | 75 fila

C:\Users\claud\AppData\Local\Temp\ipykernel_36180\3211343472.py:160: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True)


,neighborhood,new,index,used,mes,pais,ciudad,operacion
0,La Planicie - El Sol de la Molina,NaN,None,NaN,abr. 17,Ecuador,Quito,sell
1,Lima Centro,NaN,None,NaN,abr. 17,Ecuador,Quito,sell
2,Brena Norte,NaN,None,NaN,abr. 17,Ecuador,Quito,sell
3,Nuevo Barranco,NaN,None,NaN,abr. 17,Ecuador,Quito,sell
4,San Borja centro,NaN,None,NaN,abr. 17,Ecuador,Quito,sell
...,...,...,...,...,...,...,...,...
6365,Chorrillos Oeste,NaN,3683,3548.0,nov. 25,Ecuador,Quito,sell
6366,Chorrillos Este,3676.0,3679,3703.0,nov. 25,Ecuador,Quito,sell
6367,Los Olivos Oeste,NaN,3511,3408.0,nov. 25,Ecuador,Quito,sell
6368,Los Olivos Centro,3792.0,3491,3328.0,nov. 25,Ecuador,Quito,sell


#### Lima
#### Renta

In [5]:
cookies = {
    'RAP_XSRF_TOKEN': 'AImk1ALJhtDxPMT1zNJVpAfmALLP6H2zcg:1774563717716',
    '__Secure-3PSID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076',
    '__Secure-3PAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    'NID': '530=gDjAWKDTV_8uQH1Ohgs2-HpHJQcITzPv6dULBY0hR6yyNB7F-2oKCtNgUIVqnBncTRuu45eOBq6_lW2D6tVU-MigHYlB0DmkJC6kWVm-wHjgui8-EbZHXGsyznuslwPcIcmzy-prrDvfd3Ru4jk4XAZIsLM2PR93WJB9vo6CIIUv3Baxc4hdYD6JqeNaUwZIGRZRYRfDB8F8L5SFRmLH4IzTGsJy8eI-2dOo8_s_lucUsmMTvyzai6sq44m9HaW6vhoWF3mmEZj0OZwJBSS4kDht1ZBaiX23cFtOK2JQziKmQCWmkmMlQxsdJWvrCLPX1nMgribsUGpjxaZh-919GqkPY3XHWXwADtC-m7xConPO6YUJLGeEVKfgt2mq5pJ7L7bhYFa_rikYJe9alMKX_qP5ZE9Pg5ygbOhHbH2HnDjvwd7vZGQFA7Zti3JnKMUA0UYAhZC2MItW6mCmdAzopnFc4MXfJpTizSs4MXedOx5Lg8uzBXgbHBSEEotmzQF-akWJIqKxsFCyYXr34cJYGGVzDabEJCo4zB4NY9txqCxzpovTfSz-J29CAqtpnPEtfe0fLoC0c82SiX2K7JOazTrPP0A3HCDvSVDQAY1ME-7kmihqvrNpmPJkgZ8pcwPJ_1p5kW285IR4Dh-WXQuBM-06SxzRpmMxaMP3d5iElNoTBIuAsJ0f0XGrU2_1GCSl4aZj3uowPiLdC5eJ7n7aW969icdLcNON9wS9p8YGWBRanBXJINsn8UxkN5OlAHQ5tmezAppFldKnaeX5BzsSKBEib_DHINUgmztxE0Xy4xYZBBU07WUVx36j-klMs8u7kp8ismKiDnSMryyzbSzBCih52um0c506UklUG0kBhsH2imhGgN4pfF2MZpp9GuQs4MbTgQOzS3jE5t_ivryStDlTbshN7m05iVvfbMqZm_fRRFsXSQKLF4Zq3aJncxYKvM8c6HL6NqU0j3h-wMMeeB77AeJePTI-8CB_zJxjTU-6BulqxurtkY7yT6y_WUoyWibop--ikipErr3HCBDTgTweQ_vRh41NQ1vvMxRiUqFsuC03OK0_S5IzByqa_HtjsaXe_e86hy1mftbM9yjPm05PCR69QkzYbfzqf75XsrXWgO6cP6OWZrFApgkSM_ROXsTbVgcpyojMCt1D_XG41CKQPnNEYRl4WRZfsdstBxggIQbLXuJfD2xpiy2QrOFP6j2QT1QakXGQEu-4q6RWdQAWmlC0T8V85lgoDw4Op-MKDOhN9IiV2I8GuRjHmWQpyfplO7fy0VjK0Qi-lRc4OW7SJMnIcWvG8CotW8Bvxk0kEKjn3zA7V0mG9B68HyF5wippZsBbHDMMbeDTnWcHftdQWOo2ss0GX3TzgiutFzOldhLZeXBDhwqQX3_0opcjwHfs2UvgRwR37x_ZE9Tp--2vvOWdhBjGU8zlo1n13XaUuZrJIxbuRTPBWcx9zq4kClqeK67slm2Y1BACt6U5r2TofmzwKNUAWneSWJQczzDly3pV1YiyuvM3YDKx1ZJzPCBYErhRlPdfFhnOxg2ol2GI41xcR1X0M_63xB9WJLHyMklcdrn3a3JMA1pvrWB7eYQKFWvobakGGA2NZArNxZ4eaH9kWuH_Bbkt0edkqT9_RzHV3BYYDw5www-CHiS79FePx2HgsqJAqlNkrmHFaTPT9isPpdwJQ9X3aan1RkSlSpYMXIjYds2KgSr5Q4fLWZk7h1U-zdvNmyF1etF_LO-eHecJzxPP_lGnQlhSWsSfLP9jl9996_o7-5SivKgWv_Q4aV0-6PKN9lupTWKQ10qSGPRcl0jA4wcw0AaTLuZfc_wkraUiEMw9n0tNQySDfxjXVzC-Fa_3mbRG1FqBQCZGA3U3932u4Es9mhf-yq7DreHX6zt2qFRGCOvpsz-7CV91y_fDJ__6pYD5A55PWtafYs3dHrziNV-8_0kiDGFLPGLrPOw_mM2BZjLTbgpWEXc5CpLjARx_oYvFU99Uqn6ljjPvwR5dPvhmL12CceVOs52AlarWKnoNV2zxzdTmbjPizwzbf4VaoAhHcOG78fL4Sjj9EbooHXbku5kfBqZGQj0axSBauC7YIEcIrAD7oiB7KLNjQ5IRUq65QiRkMoPKQvfhUcioET-V',
    '__Secure-3PSIDTS': 'sidts-CjEBWhotCQJP50JpyGJfg879V0TXwvSagsd1A9gkGTVKcoigQ8NU2W6Po8DnUR8NngGDEAA',
    '__Secure-3PSIDCC': 'AKEyXzWXquuiGgfhfMvJkUhEUughSuRADTOo0lqimyAfbjLlRC70BbovVSUCygSqli47xeKiWC9z9w',
}

headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'es-US,es-419;q=0.9,es;q=0.8',
    'content-type': 'application/json',
    'encoding': 'null',
    'origin': 'https://lookerstudio.google.com',
    'priority': 'u=1, i',
    'referer': 'https://lookerstudio.google.com/embed/u/0/reporting/22a54766-3bba-4dc8-9735-2aebe427679c/page/Ibx9',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'sec-fetch-storage-access': 'active',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-browser-channel': 'stable',
    'x-browser-copyright': 'Copyright 2026 Google LLC. All Rights reserved.',
    'x-browser-validation': 'NNdAQ4umZKwG7+IzfjmwHCgaZsw=',
    'x-browser-year': '2026',
    'x-client-data': 'CJC2yQEIo7bJAQipncoBCNmOywEIk6HLAQiFoM0BGLGKzwEY7LHPAQ==',
    'x-rap-xsrf-token': 'AImk1ALJhtDxPMT1zNJVpAfmALLP6H2zcg:1774563717716',
    # 'cookie': 'RAP_XSRF_TOKEN=AImk1ALJhtDxPMT1zNJVpAfmALLP6H2zcg:1774563717716; __Secure-3PSID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076; __Secure-3PAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; NID=530=gDjAWKDTV_8uQH1Ohgs2-HpHJQcITzPv6dULBY0hR6yyNB7F-2oKCtNgUIVqnBncTRuu45eOBq6_lW2D6tVU-MigHYlB0DmkJC6kWVm-wHjgui8-EbZHXGsyznuslwPcIcmzy-prrDvfd3Ru4jk4XAZIsLM2PR93WJB9vo6CIIUv3Baxc4hdYD6JqeNaUwZIGRZRYRfDB8F8L5SFRmLH4IzTGsJy8eI-2dOo8_s_lucUsmMTvyzai6sq44m9HaW6vhoWF3mmEZj0OZwJBSS4kDht1ZBaiX23cFtOK2JQziKmQCWmkmMlQxsdJWvrCLPX1nMgribsUGpjxaZh-919GqkPY3XHWXwADtC-m7xConPO6YUJLGeEVKfgt2mq5pJ7L7bhYFa_rikYJe9alMKX_qP5ZE9Pg5ygbOhHbH2HnDjvwd7vZGQFA7Zti3JnKMUA0UYAhZC2MItW6mCmdAzopnFc4MXfJpTizSs4MXedOx5Lg8uzBXgbHBSEEotmzQF-akWJIqKxsFCyYXr34cJYGGVzDabEJCo4zB4NY9txqCxzpovTfSz-J29CAqtpnPEtfe0fLoC0c82SiX2K7JOazTrPP0A3HCDvSVDQAY1ME-7kmihqvrNpmPJkgZ8pcwPJ_1p5kW285IR4Dh-WXQuBM-06SxzRpmMxaMP3d5iElNoTBIuAsJ0f0XGrU2_1GCSl4aZj3uowPiLdC5eJ7n7aW969icdLcNON9wS9p8YGWBRanBXJINsn8UxkN5OlAHQ5tmezAppFldKnaeX5BzsSKBEib_DHINUgmztxE0Xy4xYZBBU07WUVx36j-klMs8u7kp8ismKiDnSMryyzbSzBCih52um0c506UklUG0kBhsH2imhGgN4pfF2MZpp9GuQs4MbTgQOzS3jE5t_ivryStDlTbshN7m05iVvfbMqZm_fRRFsXSQKLF4Zq3aJncxYKvM8c6HL6NqU0j3h-wMMeeB77AeJePTI-8CB_zJxjTU-6BulqxurtkY7yT6y_WUoyWibop--ikipErr3HCBDTgTweQ_vRh41NQ1vvMxRiUqFsuC03OK0_S5IzByqa_HtjsaXe_e86hy1mftbM9yjPm05PCR69QkzYbfzqf75XsrXWgO6cP6OWZrFApgkSM_ROXsTbVgcpyojMCt1D_XG41CKQPnNEYRl4WRZfsdstBxggIQbLXuJfD2xpiy2QrOFP6j2QT1QakXGQEu-4q6RWdQAWmlC0T8V85lgoDw4Op-MKDOhN9IiV2I8GuRjHmWQpyfplO7fy0VjK0Qi-lRc4OW7SJMnIcWvG8CotW8Bvxk0kEKjn3zA7V0mG9B68HyF5wippZsBbHDMMbeDTnWcHftdQWOo2ss0GX3TzgiutFzOldhLZeXBDhwqQX3_0opcjwHfs2UvgRwR37x_ZE9Tp--2vvOWdhBjGU8zlo1n13XaUuZrJIxbuRTPBWcx9zq4kClqeK67slm2Y1BACt6U5r2TofmzwKNUAWneSWJQczzDly3pV1YiyuvM3YDKx1ZJzPCBYErhRlPdfFhnOxg2ol2GI41xcR1X0M_63xB9WJLHyMklcdrn3a3JMA1pvrWB7eYQKFWvobakGGA2NZArNxZ4eaH9kWuH_Bbkt0edkqT9_RzHV3BYYDw5www-CHiS79FePx2HgsqJAqlNkrmHFaTPT9isPpdwJQ9X3aan1RkSlSpYMXIjYds2KgSr5Q4fLWZk7h1U-zdvNmyF1etF_LO-eHecJzxPP_lGnQlhSWsSfLP9jl9996_o7-5SivKgWv_Q4aV0-6PKN9lupTWKQ10qSGPRcl0jA4wcw0AaTLuZfc_wkraUiEMw9n0tNQySDfxjXVzC-Fa_3mbRG1FqBQCZGA3U3932u4Es9mhf-yq7DreHX6zt2qFRGCOvpsz-7CV91y_fDJ__6pYD5A55PWtafYs3dHrziNV-8_0kiDGFLPGLrPOw_mM2BZjLTbgpWEXc5CpLjARx_oYvFU99Uqn6ljjPvwR5dPvhmL12CceVOs52AlarWKnoNV2zxzdTmbjPizwzbf4VaoAhHcOG78fL4Sjj9EbooHXbku5kfBqZGQj0axSBauC7YIEcIrAD7oiB7KLNjQ5IRUq65QiRkMoPKQvfhUcioET-V; __Secure-3PSIDTS=sidts-CjEBWhotCQJP50JpyGJfg879V0TXwvSagsd1A9gkGTVKcoigQ8NU2W6Po8DnUR8NngGDEAA; __Secure-3PSIDCC=AKEyXzWXquuiGgfhfMvJkUhEUughSuRADTOo0lqimyAfbjLlRC70BbovVSUCygSqli47xeKiWC9z9w',
}

params = {
    'appVersion': '20260315_0000',
}

json_data = {
    'dataRequest': [
        {
            'requestContext': {
                'reportContext': {
                    'reportId': '22a54766-3bba-4dc8-9735-2aebe427679c',
                    'pageId': '14728046',
                    'mode': 1,
                    'componentId': 'cd-75b23orndc',
                    'displayType': 'simple-table',
                    'actionId': 'crossFilters',
                },
                'requestMode': 0,
            },
            'datasetSpec': {
                'dataset': [
                    {
                        'datasourceId': '8195b1fb-609b-4c7d-afc8-88810deb3e48',
                        'revisionNumber': 0,
                        'parameterOverrides': [],
                    },
                ],
                'queryFields': [
                    {
                        'name': 'qt_imb9k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_nombre_n3_',
                        },
                    },
                    {
                        'name': 'qt_oz68k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_estrenar_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_1a98k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_index_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_u298k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_usado_precio_',
                            'aggregation': 1,
                        },
                    },
                ],
                'sortData': [
                    {
                        'sortColumn': {
                            'name': 'qt_1a98k9zjic',
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'dataTransformation': {
                                'sourceFieldName': '_index_precio_',
                                'aggregation': 1,
                            },
                        },
                        'sortDir': 1,
                    },
                ],
                'includeRowsCount': True,
                'relatedDimensionMask': {
                    'addDisplay': False,
                    'addUniqueId': False,
                    'addLatLong': False,
                },
                'paginateInfo': {
                    'startRow': 1,
                    'rowsCount': 1000,
                },
                'dsFilterOverrides': [],
                'filters': [
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': False,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_ho99vjcxac',
                                },
                                'filterConditionType': 'NU',
                                'stringValues': [
                                    '',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_nombre_n3_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_6dd35chjhc',
                                },
                                'filterConditionType': 'EQ',
                                'stringValues': [
                                    'LIMA',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_ciudad_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'name': 'qt_lk69k9zjic',
                                    'ns': 't0',
                                },
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_fecha_',
                                    },
                                },
                                'filterConditionType': 'IN',
                                'stringValues': [
                                    'ago 25',
                                ],
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                        'isCanvasFilter': True,
                    },
                ],
                'features': [],
                'dateRanges': [],
                'contextNsCount': 1,
                'dateRangeDimensions': [
                    {
                        'name': 'qt_b1g9k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_fecha_as_date_',
                            'transformationConfig': {
                                'transformationType': 5,
                            },
                        },
                    },
                ],
                'calculatedField': [],
                'needGeocoding': False,
                'geoFieldMask': [],
                'multipleGeocodeFields': [],
                'timezone': 'America/Mexico_City',
            },
            'role': 'main',
            'retryHints': {
                'useClientControlledRetry': True,
                'isLastRetry': False,
                'retryCount': 0,
                'originalRequestId': 'cd-75b23orndc_0_0',
            },
        },
    ],
}

In [6]:
lima_renta = run_looker_pipeline(cookies, headers, params, json_data, pais="Peru",ciudad="Lima",operacion="sell", 
                                mes_field_name="_fecha_",
                                 
                                 columnas={"col_0":"neighborhood",
                                                                                                                                     "col_1":"new",
                                                                                                                                     "col_2": "index",
                                                                                                                                     "col_3": "used"
                                                                                                                                     })
print("Total de registros", len(lima_renta))
lima_renta.to_csv("lima_renta.csv", encoding="latin1",index=False)
lima_renta

📍 Usando filtro de mes en índice: 2
⚠️ ene. 15 sin datos
⚠️ feb. 15 sin datos
⚠️ mar. 15 sin datos
⚠️ abr. 15 sin datos
⚠️ may. 15 sin datos
⚠️ jun. 15 sin datos
⚠️ jul. 15 sin datos
⚠️ ago. 15 sin datos
⚠️ sep. 15 sin datos
⚠️ oct. 15 sin datos
⚠️ nov. 15 sin datos
⚠️ dic. 15 sin datos
⚠️ ene. 16 sin datos
⚠️ feb. 16 sin datos
⚠️ mar. 16 sin datos
⚠️ abr. 16 sin datos
⚠️ may. 16 sin datos
⚠️ jun. 16 sin datos
⚠️ jul. 16 sin datos
⚠️ ago. 16 sin datos
⚠️ sep. 16 sin datos
⚠️ oct. 16 sin datos
⚠️ nov. 16 sin datos
⚠️ dic. 16 sin datos
⚠️ ene. 17 sin datos
⚠️ feb. 17 sin datos
⚠️ mar. 17 sin datos
✅ abr. 17 | 65 filas
✅ may. 17 | 65 filas
✅ jun. 17 | 65 filas
✅ jul. 17 | 65 filas
✅ ago. 17 | 65 filas
✅ sep. 17 | 65 filas
✅ oct. 17 | 65 filas
✅ nov. 17 | 65 filas
✅ dic. 17 | 62 filas
✅ ene. 18 | 65 filas
✅ feb. 18 | 64 filas
✅ mar. 18 | 64 filas
✅ abr. 18 | 65 filas
✅ may. 18 | 64 filas
✅ jun. 18 | 65 filas
✅ jul. 18 | 65 filas
✅ ago. 18 | 65 filas
✅ sep. 18 | 65 filas
✅ oct. 18 | 65 fila

,neighborhood,new,index,used,mes,pais,ciudad,operacion
0,Barranco Cultural,NaN,3585.0,3454.5,abr. 17,Peru,Lima,sell
1,Nuevo Barranco,NaN,3563.5,3662.5,abr. 17,Peru,Lima,sell
2,San Isidro Olivar,3442.0,3331.0,3317.5,abr. 17,Peru,Lima,sell
3,Pueblo Libre Este,NaN,3285.0,3273.0,abr. 17,Peru,Lima,sell
4,Miraflores Sureste,4604.5,3216.5,3156.0,abr. 17,Peru,Lima,sell
...,...,...,...,...,...,...,...,...
5521,Chorrillos Este,NaN,2330.0,NaN,nov. 25,Peru,Lima,sell
5522,Surco Este,2064.0,2181.0,2207.0,nov. 25,Peru,Lima,sell
5523,Los Olivos Sur,NaN,1816.0,1578.0,nov. 25,Peru,Lima,sell
5524,Los Olivos Centro,NaN,1490.0,1482.0,nov. 25,Peru,Lima,sell
